# Clean Charges Pipeline
Go through the `Charges` column, clean it, extract statutes, split multiple charges into separate columns, and save as checkpoint 13.

**Steps:**
1. Load data and filter to rows with charges
2. Remove known junk entries (e.g. "LAWRENCE, MA")
3. Detect and flag noise / suspicious entries
4. Standardize text (lowercase, strip whitespace, etc.)
5. Extract statute references into a separate column
6. Split multiple charges into `charge_1`, `charge_2`, ... columns
7. Save cleaned checkpoint and a unique charges reference CSV

### Imports

In [3]:
import os
import re
import numpy as np
import pandas as pd

### Set up file paths
Define where the input checkpoint lives and where outputs will be saved.

In [ ]:
NOTEBOOK_DIR = os.getcwd()
DATA_DIR = os.path.join(NOTEBOOK_DIR, "..", "..", "data")
CHECKPOINTS_DIR = os.path.join(DATA_DIR, "checkpoints")
CHARGES_DIR = os.path.join(DATA_DIR, "charges")
DATA_PATH = os.path.join(CHECKPOINTS_DIR, "checkpoint12_age.csv")
OUT_PATH = os.path.join(CHECKPOINTS_DIR, "checkpoint13_cleaned_charges.csv")
UNIQUE_CHARGES_PATH = os.path.join(CHARGES_DIR, "unique_charges_reference.csv")

# Create output directories if they don't exist
os.makedirs(CHARGES_DIR, exist_ok=True)

print("Input:", DATA_PATH)
print("Output:", OUT_PATH)
print("Unique charges ref:", UNIQUE_CHARGES_PATH)

### Load checkpoint 12 and preview
Load the CSV and take a first look at the shape and the `Charges` column.

In [5]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nCharges non-null count:", df["Charges"].notna().sum())
print("Charges null count:", df["Charges"].isna().sum())
df.head()

Shape: (424273, 16)

Columns: ['Date', 'Type', 'Location', 'Arrested', 'Location Prefix', 'DOB', 'Charges', 'latitude', 'longitude', 'Cleaned Location', 'person_id', 'category_archive', 'Year', 'crime_severity', 'category', 'Age']

Charges non-null count: 5009
Charges null count: 419264


,Date,Type,Location,Arrested,Location Prefix,DOB,Charges,latitude,longitude,Cleaned Location,person_id,category_archive,Year,crime_severity,category,Age
0,2018-01-01 00:01:00,NOISE ORD,3 HARRIMAN ST,Yes,NaN,1974-07-03,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,42.718522,-71.148148,3 HARRIMAN ST,20f72481f083d4756c89b98fd499514c5953a8a4c10253...,PUBLIC_DISTURBANCES,2018,Non-Serious,Public Disturbances,43.0
1,2018-01-01 00:08:00,LOUD NOISE,1 HARRIMAN ST FL 2,Yes,NaN,1979-01-21,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,42.718522,-71.148148,1 HARRIMAN ST,2d5adb553cad8a22fd72d2e390525997c4963bac1b185e...,PUBLIC_DISTURBANCES,2018,Non-Serious,Public Disturbances,38.0
2,2018-01-01 00:11:00,ALARM/BURG,16 ALLEN ST,No,MATOS,NaN,NaN,42.710782,-71.151911,16 ALLEN ST,NaN,FIRE_AND_ARSON_INCIDENTS,2018,Non-Serious,Fire and Arson Incidents,NaN
3,2018-01-01 00:14:00,DISORDERLY,11 SUMMER ST,No,NaN,NaN,NaN,42.711117,-71.153015,11 SUMMER ST,NaN,PUBLIC_DISTURBANCES,2018,Non-Serious,Public Disturbances,NaN
4,2018-01-01 00:27:00,EXTRA SURVEIL,57 SPRINGFIELD ST,Yes,WARD SIX CLUB,2002-06-26,A&B DOMESTIC NO 209A IN EFFECT,42.699339,-71.156938,57 SPRINGFIELD ST,41c43f9f4255dee8e2c910e87f2a983e1b13beecf27eac...,PREVENTIVE_POLICING,2018,Non-Serious,Preventive Policing,15.0


### Filter to rows that have non-empty charges
Keep only rows where `Charges` is not null and not blank. We keep a copy of the full dataframe so we can merge back later.

In [6]:
df_full = df.copy()

mask = df["Charges"].notna() & (df["Charges"].astype(str).str.strip() != "")
df = df[mask].copy().reset_index(drop=True)

print("Rows with charges:", df.shape[0])
print("Rows without charges (kept in df_full):", df_full.shape[0] - df.shape[0])
df["Charges"].head(10)

Rows with charges: 5009
Rows without charges (kept in df_full): 419264


0    A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...
1    A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...
2                       A&B DOMESTIC NO 209A IN EFFECT
3    USE MV WITHOUT AUTHORITY c90 S24; LARCENY UNDE...
4    WITNESS, INTIMIDATE c268 S13B; FALSE NAME/SS# ...
5    TRAFFICKING CLASS B: 28+ GRAMS C94 S32E; DRUG,...
6                       DRUG, POSSESS CLASS B c94C S34
7                       DRUG, POSSESS CLASS B c94C S34
8                LICENSE SUSPENDED, OP MV WITH c90 S23
9    LICENSE REVOKED AS HTO, OPERATE MV WITH c90 S2...
Name: Charges, dtype: object

### Remove known junk: "LAWRENCE, MA" variants
Some entries in the Charges column are just the city/state "LAWRENCE, MA" with various spacing. These are not real charges — set them to NaN.

In [7]:
# Match "LAWRENCE, MA", "LAWRENCE MA", "LAWRENCE,MA", "LAWRENCE , MA" etc.
lawrence_pattern = r"^\s*LAWRENCE\s*,?\s*MA\s*$"

lawrence_mask = df["Charges"].astype(str).str.contains(lawrence_pattern, case=False, na=False)
print(f"Entries matching 'LAWRENCE, MA' pattern: {lawrence_mask.sum()}")
print("\nSample of matched entries:")
print(df.loc[lawrence_mask, "Charges"].value_counts())

df.loc[lawrence_mask, "Charges"] = np.nan

Entries matching 'LAWRENCE, MA' pattern: 440

Sample of matched entries:
Charges
LAWRENCE, MA     435
LAWRENCE , MA      5
Name: count, dtype: int64


### Detect suspicious / noisy entries
Look for entries that probably aren't real charges:
- Very short entries (under 3 characters)
- Purely numeric entries

We also scan for inline junk (URLs, dates, "PublicLog" artifacts) embedded in otherwise valid charges — these will be stripped out rather than deleting the whole entry.

In [8]:
charges_remaining = df["Charges"].dropna().astype(str)

# Very short entries (under 3 characters after stripping)
short_mask = charges_remaining.str.strip().str.len() < 3
short_entries = charges_remaining[short_mask]
print(f"=== Short entries (< 3 chars): {len(short_entries)} ===")
if len(short_entries) > 0:
    print(short_entries.value_counts())

# Purely numeric entries
numeric_mask = charges_remaining.str.strip().str.match(r"^\d+$")
numeric_entries = charges_remaining[numeric_mask]
print(f"\n=== Purely numeric entries: {len(numeric_entries)} ===")
if len(numeric_entries) > 0:
    print(numeric_entries.value_counts())

# Check for entries with inline junk (URLs, dates, "PublicLog" text)
# These are valid charges but have artifacts that need stripping
junk_pattern = r"\d+\.\d+\.\d+\.\d+|PublicLog|Page \d+ of \d+|\d{1,2}/\d{1,2}/\d{2,4}"
junk_mask = charges_remaining.str.contains(junk_pattern, case=False, na=False)
junk_entries = charges_remaining[junk_mask]
print(f"\n=== Entries with inline junk (URLs/dates/PublicLog) to strip: {len(junk_entries)} ===")
if len(junk_entries) > 0:
    print(junk_entries.value_counts())

print(f"\n=== Total entries to DELETE (short + numeric): {(short_mask | numeric_mask).sum()} ===")

=== Short entries (< 3 chars): 0 ===

=== Purely numeric entries: 0 ===

=== Entries with inline junk (URLs/dates/PublicLog) to strip: 235 ===
Charges
DEFAULT WARRANT: DRUG, POSSESS CLASS B c94C S34;  2/5/2020; QuickSearch Results Page 8 of 14; DEFAULT WARRANT: LARCENY UNDER $1200 c266 S30(1)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

### Clean up noise
- **Delete** short and purely numeric entries (set to NaN)
- **Strip** inline junk (URLs, IP addresses, dates, "PublicLog Page X of Y") from otherwise valid charges

In [9]:
charges_series = df["Charges"].astype(str)
is_not_null = df["Charges"].notna()

# Delete short and numeric entries
noise_short = is_not_null & (charges_series.str.strip().str.len() < 3)
noise_numeric = is_not_null & charges_series.str.strip().str.match(r"^\d+$")
noise_combined = noise_short | noise_numeric
print(f"Entries deleted (short + numeric): {noise_combined.sum()}")
df.loc[noise_combined, "Charges"] = np.nan

# Strip inline junk from valid charges (URLs, IPs, dates, PublicLog artifacts)
junk_patterns = [
    r"\d+\.\d+\.\d+\.\d+/\S*",        # IP addresses and URL paths
    r"PublicLog\s*Page\s*\d+\s*of\s*\d+",  # "PublicLog Page 3 of 14"
    r"PublicLog",                        # standalone "PublicLog"
    r"QuickSearch\s*Results\s*Page\s*\d+\s*of\s*\d+",  # "QuickSearch Results Page 8 of 14"
    r"\d{1,2}/\d{1,2}/\d{2,4},?\s*\d{1,2}:\d{2}\s*(AM|PM)?",  # dates with times like "5/26/24, 12:25 AM"
    r"\d{1,2}/\d{1,2}/\d{2,4}",        # standalone dates like "10/18/2022"
]

mask_valid = df["Charges"].notna()
for pattern in junk_patterns:
    df.loc[mask_valid, "Charges"] = (
        df.loc[mask_valid, "Charges"]
        .astype(str)
        .str.replace(pattern, "", regex=True, case=False)
    )

# Clean up artifacts left behind (extra semicolons, spaces)
df.loc[mask_valid, "Charges"] = (
    df.loc[mask_valid, "Charges"]
    .str.replace(r";\s*;", ";", regex=True)
    .str.replace(r"^\s*;\s*|\s*;\s*$", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# If stripping junk left an empty string, set to NaN
# (e.g. entries that were ONLY "9/2/2019" or "PublicLog Page 11 of 15")
empty_after_strip = mask_valid & (df["Charges"].astype(str).str.strip() == "")
df.loc[empty_after_strip, "Charges"] = np.nan
print(f"Entries that became empty after junk stripping (set to NaN): {empty_after_strip.sum()}")

print(f"Remaining non-null charges: {df['Charges'].notna().sum()}")

Entries deleted (short + numeric): 0
Entries that became empty after junk stripping (set to NaN): 19
Remaining non-null charges: 4550


### Standardize charge text
- Convert to lowercase for case insensitivity
- Strip leading/trailing whitespace
- Remove trailing semicolons
- Collapse multiple spaces into one

In [10]:
mask_valid = df["Charges"].notna()

df.loc[mask_valid, "Charges"] = (
    df.loc[mask_valid, "Charges"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r";\s*$", "", regex=True)   # remove trailing semicolons
    .str.replace(r"\s+", " ", regex=True)     # collapse multiple spaces
    .str.strip()
)

print("Sample standardized charges:")
print(df.loc[mask_valid, "Charges"].head(10))
print(f"\nTotal non-null: {mask_valid.sum()}")

Sample standardized charges:
0    a&b on family / household member / intimate pa...
1    a&b on family / household member / intimate pa...
2                       a&b domestic no 209a in effect
3    use mv without authority c90 s24; larceny unde...
4    witness, intimidate c268 s13b; false name/ss# ...
5    trafficking class b: 28+ grams c94 s32e; drug,...
6                       drug, possess class b c94c s34
7                       drug, possess class b c94c s34
8                license suspended, op mv with c90 s23
9    license revoked as hto, operate mv with c90 s2...
Name: Charges, dtype: object

Total non-null: 4550


### Extract statute references
Statutes appear inline with charge text in patterns like `c90 S24`, `c94C S32A`, `C266 S126A`, or sometimes without the `c` prefix like `265 S15B`.

We extract all statute references into a `statutes` column and create a `cleaned_charges` column with statutes removed.

In [11]:
# Pattern for statutes: optional 'c' + digits + optional letter + space + 's' + digits + optional letters
statute_pattern = r"c?\d+[a-z]?\s+s\d+[a-z]*"

def extract_statutes(charge_str):
    """Extract all statute references from a charge string."""
    if pd.isna(charge_str):
        return np.nan
    matches = re.findall(statute_pattern, str(charge_str), re.IGNORECASE)
    if matches:
        return "; ".join(m.strip() for m in matches)
    return np.nan

def remove_statutes(charge_str):
    """Remove statute references from a charge string and clean up."""
    if pd.isna(charge_str):
        return np.nan
    cleaned = re.sub(statute_pattern, "", str(charge_str), flags=re.IGNORECASE)
    # Clean up leftover artifacts: extra spaces, dangling semicolons
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    cleaned = re.sub(r";\s*;", ";", cleaned)  # collapse double semicolons
    cleaned = re.sub(r"^;\s*|\s*;$", "", cleaned).strip()  # strip leading/trailing semicolons
    return cleaned if cleaned else np.nan

df["statutes"] = df["Charges"].apply(extract_statutes)
df["cleaned_charges"] = df["Charges"].apply(remove_statutes)

print("Sample results:")
sample = df[df["statutes"].notna()][["Charges", "statutes", "cleaned_charges"]].head(10)
print(sample.to_string())

print(f"\nRows with statutes: {df['statutes'].notna().sum()}")
print(f"Rows without statutes: {df['statutes'].isna().sum()}")

Sample results:
                                                                                                                                                                                                                            Charges                                 statutes                                                                                                                                                                                 cleaned_charges
3                                                                                                                                                                     use mv without authority c90 s24; larceny under $250 c266 s30                        c90 s24; c266 s30                                                                                                                                                   use mv without authority ; larceny under $250
4   witness, intimidate c268 s13b; false name/ss# to law enfor

### Preview cleaned charges
See what the cleaned charge text looks like and check for any remaining issues.

In [12]:
print("Top 30 most common cleaned charges:")
print(df["cleaned_charges"].value_counts().head(30))

print(f"\nTotal unique cleaned charges: {df['cleaned_charges'].nunique()}")
print(f"\nNull cleaned_charges: {df['cleaned_charges'].isna().sum()}")

Top 30 most common cleaned charges:
cleaned_charges
drug, possess class a                                  122
sexual conduct for fee                                  93
unlicensed operation of mv                              68
disorderly conduct                                      64
drug, possess class b                                   64
license suspended, op mv with                           57
any person trafficks fentanyl more than 10 grams        51
oui liquor                                              49
drug, distribute class a                                48
trespass                                                46
drinking in public                                      45
fugitive from justice on court warrant                  45
trafficking class a: 28+ grams                          37
drug, possess class a ; drug, possess class b           33
motor veh, receive stolen                               32
shoplifting by asportation                              31
assa

### Split multiple charges into separate columns
Each row can have multiple charges separated by semicolons. Split them into `charge_1`, `charge_2`, ..., `charge_N` columns. Do the same for statutes.

In [13]:
# Split cleaned_charges by semicolon
charge_split = (
    df["cleaned_charges"]
    .fillna("")
    .str.split(";")
    .apply(lambda parts: [p.strip() for p in parts if p.strip()])
)

# Find the maximum number of charges in any row
max_charges = charge_split.apply(len).max()
print(f"Maximum charges in a single row: {max_charges}")

# Create charge_1 through charge_N columns
for i in range(max_charges):
    col_name = f"charge_{i+1}"
    df[col_name] = charge_split.apply(lambda x, idx=i: x[idx] if idx < len(x) else np.nan)

# Split statutes by semicolon
statute_split = (
    df["statutes"]
    .fillna("")
    .str.split(";")
    .apply(lambda parts: [p.strip() for p in parts if p.strip()])
)

max_statutes = statute_split.apply(len).max()
print(f"Maximum statutes in a single row: {max_statutes}")

for i in range(max_statutes):
    col_name = f"statute_{i+1}"
    df[col_name] = statute_split.apply(lambda x, idx=i: x[idx] if idx < len(x) else np.nan)

print(f"\nNew columns added: charge_1 to charge_{max_charges}, statute_1 to statute_{max_statutes}")
print(f"Total columns now: {len(df.columns)}")
df[["cleaned_charges", "charge_1", "charge_2", "charge_3", "statutes", "statute_1", "statute_2"]].head(10)

Maximum charges in a single row: 29
Maximum statutes in a single row: 21

New columns added: charge_1 to charge_29, statute_1 to statute_21
Total columns now: 68


,cleaned_charges,charge_1,charge_2,charge_3,statutes,statute_1,statute_2
0,a&b on family / household member / intimate pa...,a&b on family / household member / intimate pa...,strangulation or suffocation,NaN,NaN,NaN,NaN
1,a&b on family / household member / intimate pa...,a&b on family / household member / intimate pa...,NaN,NaN,NaN,NaN,NaN
2,a&b domestic no 209a in effect,a&b domestic no 209a in effect,NaN,NaN,NaN,NaN,NaN
3,use mv without authority ; larceny under $250,use mv without authority,larceny under $250,NaN,c90 s24; c266 s30,c90 s24,c266 s30
4,"witness, intimidate ; false name/ss# to law en...","witness, intimidate",false name/ss# to law enforcement,threat to commit crime,c268 s13b; c275 s2; c265 s15b; c265 s26,c268 s13b,c275 s2
5,"trafficking class b: 28+ grams ; drug, distrib...",trafficking class b: 28+ grams,"drug, distribute class b","drug, possess to distrib class b",c94 s32e; c94c s32a; c94c s32a,c94 s32e,c94c s32a
6,"drug, possess class b","drug, possess class b",NaN,NaN,c94c s34,c94c s34,NaN
7,"drug, possess class b","drug, possess class b",NaN,NaN,c94c s34,c94c s34,NaN
8,"license suspended, op mv with","license suspended, op mv with",NaN,NaN,c90 s23,c90 s23,NaN
9,"license revoked as hto, operate mv with ; stop...","license revoked as hto, operate mv with","stop/yield, fail to","signal, fail to",c90 s23; c89 s9; c90 s14b,c90 s23,c89 s9


### Build unique charges reference
Collect all individual charge names across all `charge_N` columns, count their occurrences, and save to `data/charges/unique_charges_reference.csv`.

In [14]:
# Gather all charge values from charge_N columns
charge_cols = [c for c in df.columns if re.match(r"^charge_\d+$", c)]
all_charges = pd.concat([df[c] for c in charge_cols], ignore_index=True).dropna()

charge_counts = all_charges.value_counts().reset_index()
charge_counts.columns = ["charge", "count"]

print(f"Total unique charges: {len(charge_counts)}")
print(f"\nTop 30 charges by frequency:")
print(charge_counts.head(30).to_string(index=False))

# Save
charge_counts.to_csv(UNIQUE_CHARGES_PATH, index=False)
print(f"\nSaved to {UNIQUE_CHARGES_PATH}")

Total unique charges: 1760

Top 30 charges by frequency:
                                          charge  count
                      unlicensed operation of mv    390
                              disorderly conduct    339
                                resisting arrest    290
                   license suspended, op mv with    288
                           drug, possess class a    260
                           drug, possess class b    210
                                      oui liquor    160
                             stop/yield, fail to    150
                                        trespass    142
                      assault w/dangerous weapon    121
any person trafficks fentanyl more than 10 grams    121
          fugitive from justice on court warrant    108
                      unregistered motor vehicle    105
                        drug, distribute class a    102
                          sexual conduct for fee     98
                          threat to commit crim

### Save checkpoint 13
Merge the cleaned charge columns back into the full dataframe (including rows that had no charges) and save.

In [15]:
# Merge cleaned columns back into full dataframe
df_out = df_full.copy()

# Create empty columns in df_out for the new fields
new_cols = ["statutes", "cleaned_charges"] + [c for c in df.columns if re.match(r"^(charge|statute)_\d+$", c)]
for col in new_cols:
    df_out[col] = np.nan

# Fill in values from df using the original index alignment
mask = df_full["Charges"].notna() & (df_full["Charges"].astype(str).str.strip() != "")
valid_indices = df_full[mask].index

for col in new_cols:
    if col in df.columns:
        df_out.loc[valid_indices, col] = df[col].values

print("Output shape:", df_out.shape)
print("\nColumns:", df_out.columns.tolist())
print(f"\nNon-null cleaned_charges: {df_out['cleaned_charges'].notna().sum()}")

# Save
df_out.to_csv(OUT_PATH, index=False)
print(f"\nSaved to {OUT_PATH}")

C:\Users\Indel\AppData\Local\Temp\ipykernel_34264\2760237554.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... 'c266 s28' 'c266 s28'
 'c266 s28; c268 s34a; c276 s19; c90 s23; c266 s28; c94c s34; c266 s30a; c266 s28; c90 s10']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[valid_indices, col] = df[col].values
C:\Users\Indel\AppData\Local\Temp\ipykernel_34264\2760237554.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['a&b on family / household member / intimate partne; strangulation or suffocation'
 'a&b on family / household member / intimate partne'
 'a&b domestic no 209a in effect' ... 'motor veh, malicious damage to'
 'motor veh, receive stolen'
 "motor veh, receive stolen ; false id info, arrst furni to law enf ; fugitive from justice on g

Output shape: (424273, 68)

Columns: ['Date', 'Type', 'Location', 'Arrested', 'Location Prefix', 'DOB', 'Charges', 'latitude', 'longitude', 'Cleaned Location', 'person_id', 'category_archive', 'Year', 'crime_severity', 'category', 'Age', 'statutes', 'cleaned_charges', 'charge_1', 'charge_2', 'charge_3', 'charge_4', 'charge_5', 'charge_6', 'charge_7', 'charge_8', 'charge_9', 'charge_10', 'charge_11', 'charge_12', 'charge_13', 'charge_14', 'charge_15', 'charge_16', 'charge_17', 'charge_18', 'charge_19', 'charge_20', 'charge_21', 'charge_22', 'charge_23', 'charge_24', 'charge_25', 'charge_26', 'charge_27', 'charge_28', 'charge_29', 'statute_1', 'statute_2', 'statute_3', 'statute_4', 'statute_5', 'statute_6', 'statute_7', 'statute_8', 'statute_9', 'statute_10', 'statute_11', 'statute_12', 'statute_13', 'statute_14', 'statute_15', 'statute_16', 'statute_17', 'statute_18', 'statute_19', 'statute_20', 'statute_21']

Non-null cleaned_charges: 4550

Saved to c:\Users\Indel\Documents\gatewayinit

### Summary

In [16]:
print("=== Cleaning Summary ===")
print(f"Total rows: {df_out.shape[0]}")
print(f"Rows with charges: {df_out['cleaned_charges'].notna().sum()}")
print(f"Rows without charges: {df_out['cleaned_charges'].isna().sum()}")
print(f"Unique charge types: {charge_counts.shape[0]}")
print(f"Max charges per row: {max_charges}")
print(f"Max statutes per row: {max_statutes}")
print(f"\nFiles saved:")
print(f"  - {OUT_PATH}")
print(f"  - {UNIQUE_CHARGES_PATH}")

=== Cleaning Summary ===
Total rows: 424273
Rows with charges: 4550
Rows without charges: 419723
Unique charge types: 1760
Max charges per row: 29
Max statutes per row: 21

Files saved:
  - c:\Users\Indel\Documents\gatewayinitiative-lawrencepd\scripts\..\data\checkpoints\checkpoint13_cleaned_charges.csv
  - c:\Users\Indel\Documents\gatewayinitiative-lawrencepd\scripts\..\data\charges\unique_charges_reference.csv


---
## Iteration 2: Fix remaining junk in charges

After reviewing `unique_charges_reference.csv`, we found ~600+ junk/fragment entries caused by:

1. **Semicolons inside statute references** (e.g. `c272; S53A` instead of `c272 S53A`) — when we split by `;`, these produce orphaned fragments like `S53A`, `with`, `license`, etc.
2. **URLs not caught** — `lawpd-qed.lawpd.cityoflawrence.com/...` domain pattern
3. **Standalone short dates** — `6/14`, `9/17`, etc.
4. **City names** still present — `lawrence, ma`, `lowell, ma`, `methuen, ma`
5. **Address entries** — `address: 155 salem st`
6. **`§` symbol** not matched by statute regex

**Strategy:** Go back to the standardized `Charges` column (before statute extraction), fix the root causes, then re-run extraction and splitting from scratch.

### Step 1: Drop iteration 1 columns and restart from Charges
Drop the `statutes`, `cleaned_charges`, and all `charge_N`/`statute_N` columns from iteration 1. We'll rebuild them after fixing the root causes.

In [17]:
# Drop all iteration 1 derived columns
cols_to_drop = ["statutes", "cleaned_charges"] + [c for c in df.columns if re.match(r"^(charge|statute)_\d+$", c)]
df = df.drop(columns=cols_to_drop, errors="ignore")

print(f"Dropped {len(cols_to_drop)} columns. Remaining columns: {df.columns.tolist()}")
print(f"Non-null Charges: {df['Charges'].notna().sum()}")

Dropped 52 columns. Remaining columns: ['Date', 'Type', 'Location', 'Arrested', 'Location Prefix', 'DOB', 'Charges', 'latitude', 'longitude', 'Cleaned Location', 'person_id', 'category_archive', 'Year', 'crime_severity', 'category', 'Age']
Non-null Charges: 4550


### Step 2: Fix semicolons inside statute references
The root cause of most fragments. Entries like `c272; S53A` should be `c272 S53A`. We normalize these BEFORE splitting by semicolon. Also normalize `§` to `S`.

In [18]:
mask_valid = df["Charges"].notna()

before = df["Charges"].copy()

# Fix semicolons inside statute references: "c272; S53A" -> "c272 S53A"
# Also handles "c94C; S32A", "c90; S24", etc.
df.loc[mask_valid, "Charges"] = (
    df.loc[mask_valid, "Charges"]
    .str.replace(r"([c¢]\d+[a-z]?)\s*;\s*([sS§$8]\d+)", r"\1 \2", regex=True, case=False)
)

# Normalize corrupted characters in statute references:
#   ¢ -> c (corrupted chapter prefix)
#   § -> S (section symbol)
# Note: $ -> S and 8 -> S only when they appear as section prefixes
# We handle $ and 8 in the statute regex itself rather than global replace
# since $ and 8 have other valid uses in charge text (dollar amounts, numbers)
df.loc[mask_valid, "Charges"] = (
    df.loc[mask_valid, "Charges"]
    .str.replace("¢", "c", regex=False)
    .str.replace("§", "S", regex=False)
)

# Count how many entries changed
changed = (before.fillna("") != df["Charges"].fillna("")).sum()
print(f"Entries fixed (semicolons in statutes + character normalization): {changed}")

# Show some examples of what changed
diff_mask = (before.fillna("") != df["Charges"].fillna(""))
if diff_mask.any():
    print("\nSample fixes (before -> after):")
    for idx in df[diff_mask].head(5).index:
        print(f"  BEFORE: {before[idx]}")
        print(f"  AFTER:  {df.loc[idx, 'Charges']}")
        print()

Entries fixed (semicolons in statutes + character normalization): 524

Sample fixes (before -> after):
  BEFORE: standard warrant: a&b c265 s13a; standard warrant: a&b on child with substantial injury c265; s13j; standard warrant: kidnapping & endanger minor by relative; c265 s26a
  AFTER:  standard warrant: a&b c265 s13a; standard warrant: a&b on child with substantial injury c265 s13j; standard warrant: kidnapping & endanger minor by relative; c265 s26a

  BEFORE: drug, possess class a c94c s34; license suspended, op mv with c90 s23; unregistered motor vehicle c90 s9; uninsured mv/trailer c90 s34j; warrant charges: default warrant: sexual conduct for fee c272; s53a; default warrant: check, utter false c267 s5; default warrant: larceny under $1200 c266 s30(1)
  AFTER:  drug, possess class a c94c s34; license suspended, op mv with c90 s23; unregistered motor vehicle c90 s9; uninsured mv/trailer c90 s34j; warrant charges: default warrant: sexual conduct for fee c272 s53a; default warran

### Step 3: Strip additional junk patterns missed in iteration 1
Remove domain URLs, standalone short dates, city/state names, address entries, and other artifacts.

In [19]:
mask_valid = df["Charges"].notna()

# Additional inline junk patterns to strip (these appear WITHIN otherwise valid charge strings)
inline_junk_patterns = [
    r"lawpd-qed\.lawpd\.cityoflawrence\.com\S*",  # domain URLs
    r"\d+\.\d+\.\d+\.\d+/\S*",                    # IP-based URLs (re-apply in case)
    r"QuickSearch\s*Results\s*Page\s*\d+\s*of\s*\d+",
    r"PublicLog\s*Page\s*\d+\s*of\s*\d+",
    r"PublicLog",
    r"\d{1,2}/\d{1,2}/\d{2,4},?\s*\d{1,2}:\d{2}\s*(AM|PM)?",  # dates with times
    r"\d{1,2}/\d{1,2}/\d{2,4}",                    # dates with year
]

for pattern in inline_junk_patterns:
    df.loc[mask_valid, "Charges"] = (
        df.loc[mask_valid, "Charges"]
        .str.replace(pattern, "", regex=True, case=False)
    )

# Clean up artifacts (extra semicolons, spaces)
df.loc[mask_valid, "Charges"] = (
    df.loc[mask_valid, "Charges"]
    .str.replace(r";\s*;", ";", regex=True)
    .str.replace(r"^\s*;\s*|\s*;\s*$", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Now handle entries that are ENTIRELY junk (set to NaN)
# These are full-entry patterns, not inline
fullentry_junk_patterns = [
    r"^\d{1,2}/\d{1,2}$",                          # short dates like "6/14"
    r"^(lawrence|lowell|methuen),?\s*ma$",           # city, state
    r"^lawrence police department$",
    r"^address:\s*.*$",                              # address entries
    r"^transferred in \[dummy.*$",                   # dummy entries
    r"^\d{1,2}/\d{4,},?\s*\d{1,2}:\d{2}\s*(am|pm)?$",  # timestamps as full entry
]

fullentry_junk_mask = pd.Series(False, index=df.index)
for pattern in fullentry_junk_patterns:
    fullentry_junk_mask = fullentry_junk_mask | (
        mask_valid & df["Charges"].str.match(pattern, case=False, na=False)
    )

print(f"Full-entry junk to set to NaN: {fullentry_junk_mask.sum()}")
if fullentry_junk_mask.any():
    print("\nEntries being removed:")
    print(df.loc[fullentry_junk_mask, "Charges"].value_counts().head(20))

df.loc[fullentry_junk_mask, "Charges"] = np.nan

# Also catch entries that became empty after stripping
empty_mask = mask_valid & (df["Charges"].astype(str).str.strip() == "")
df.loc[empty_mask, "Charges"] = np.nan
print(f"Entries empty after stripping: {empty_mask.sum()}")

print(f"\nRemaining non-null charges: {df['Charges'].notna().sum()}")

Full-entry junk to set to NaN: 4

Entries being removed:
Charges
methuen, ma    1
lowell, ma     1
6/17           1
8/13           1
Name: count, dtype: int64
Entries empty after stripping: 0

Remaining non-null charges: 4546


### Step 4: Re-run statute extraction
Now that semicolons in statutes are fixed and `§` is normalized, re-extract statutes and create `cleaned_charges`. The regex also includes `§` as an alias for `S`.

In [20]:
# Updated statute pattern — handles corrupted characters from OCR/copy:
#   Chapter prefix: c, C, ¢ (¢ is corrupted c)
#   Section prefix: s, S, §, $, 8 ($ and 8 are corrupted S)
# Pattern: [chapter_prefix]digits[optional_letter] space(s) [section_prefix]digits[optional_letters][optional_parentheticals]
statute_pattern_v2 = r"[c¢]?\d+[a-z]?\s+[sS§$8]\d+[a-z]*(?:\([a-z0-9]+\))*"

def extract_statutes_v2(charge_str):
    """Extract all statute references from a charge string."""
    if pd.isna(charge_str):
        return np.nan
    matches = re.findall(statute_pattern_v2, str(charge_str), re.IGNORECASE)
    if matches:
        return "; ".join(m.strip() for m in matches)
    return np.nan

def remove_statutes_v2(charge_str):
    """Remove statute references from a charge string and clean up."""
    if pd.isna(charge_str):
        return np.nan
    cleaned = re.sub(statute_pattern_v2, "", str(charge_str), flags=re.IGNORECASE)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    cleaned = re.sub(r";\s*;", ";", cleaned)
    cleaned = re.sub(r"^;\s*|\s*;$", "", cleaned).strip()
    return cleaned if cleaned else np.nan

df["statutes"] = df["Charges"].apply(extract_statutes_v2)
df["cleaned_charges"] = df["Charges"].apply(remove_statutes_v2)

print("Sample results:")
sample = df[df["statutes"].notna()][["Charges", "statutes", "cleaned_charges"]].head(10)
print(sample.to_string())

print(f"\nRows with statutes: {df['statutes'].notna().sum()}")
print(f"Rows without statutes: {df['statutes'].isna().sum()}")

# Verify the problematic patterns are now caught
test_patterns = ["¢265", "$17", "834a", "832e", "824", "$32a"]
for pat in test_patterns:
    hits = df["Charges"].dropna().str.contains(pat, na=False).sum()
    hits_cleaned = df["cleaned_charges"].dropna().str.contains(pat, na=False).sum()
    print(f"  '{pat}' in Charges: {hits}, in cleaned_charges: {hits_cleaned}")

Sample results:
                                                                                                                                                                                                                            Charges                                 statutes                                                                                                                                                                                 cleaned_charges
3                                                                                                                                                                     use mv without authority c90 s24; larceny under $250 c266 s30                        c90 s24; c266 s30                                                                                                                                                   use mv without authority ; larceny under $250
4   witness, intimidate c268 s13b; false name/ss# to law enfor

### Step 5: Re-split into separate columns
Split `cleaned_charges` and `statutes` by semicolon into individual columns again.

In [21]:
# Split cleaned_charges by semicolon
charge_split = (
    df["cleaned_charges"]
    .fillna("")
    .str.split(";")
    .apply(lambda parts: [p.strip() for p in parts if p.strip()])
)

max_charges = charge_split.apply(len).max()
print(f"Maximum charges in a single row: {max_charges}")

for i in range(max_charges):
    col_name = f"charge_{i+1}"
    df[col_name] = charge_split.apply(lambda x, idx=i: x[idx] if idx < len(x) else np.nan)

# Split statutes by semicolon
statute_split = (
    df["statutes"]
    .fillna("")
    .str.split(";")
    .apply(lambda parts: [p.strip() for p in parts if p.strip()])
)

max_statutes = statute_split.apply(len).max()
print(f"Maximum statutes in a single row: {max_statutes}")

for i in range(max_statutes):
    col_name = f"statute_{i+1}"
    df[col_name] = statute_split.apply(lambda x, idx=i: x[idx] if idx < len(x) else np.nan)

print(f"\nColumns: charge_1 to charge_{max_charges}, statute_1 to statute_{max_statutes}")
df[["cleaned_charges", "charge_1", "charge_2", "charge_3"]].head(10)

Maximum charges in a single row: 28
Maximum statutes in a single row: 23

Columns: charge_1 to charge_28, statute_1 to statute_23


,cleaned_charges,charge_1,charge_2,charge_3
0,a&b on family / household member / intimate pa...,a&b on family / household member / intimate pa...,strangulation or suffocation,NaN
1,a&b on family / household member / intimate pa...,a&b on family / household member / intimate pa...,NaN,NaN
2,a&b domestic no 209a in effect,a&b domestic no 209a in effect,NaN,NaN
3,use mv without authority ; larceny under $250,use mv without authority,larceny under $250,NaN
4,"witness, intimidate ; false name/ss# to law en...","witness, intimidate",false name/ss# to law enforcement,threat to commit crime
5,"trafficking class b: 28+ grams ; drug, distrib...",trafficking class b: 28+ grams,"drug, distribute class b","drug, possess to distrib class b"
6,"drug, possess class b","drug, possess class b",NaN,NaN
7,"drug, possess class b","drug, possess class b",NaN,NaN
8,"license suspended, op mv with","license suspended, op mv with",NaN,NaN
9,"license revoked as hto, operate mv with ; stop...","license revoked as hto, operate mv with","stop/yield, fail to","signal, fail to"


### Step 6: Post-split fragment cleanup
After splitting, some individual `charge_N` entries are still fragments — broken pieces of charges that got separated. Clean these out by matching known fragment patterns and setting them to NaN.

In [22]:
charge_cols = [c for c in df.columns if re.match(r"^charge_\d+$", c)]

# Fragment patterns to remove from individual charge cells
fragment_patterns = [
    r"^[a-z]$",                          # single letters
    r"^.{1,2}$",                          # anything 1-2 chars
    r"^s\d+[a-z]*(\([a-z0-9]+\))?$",     # orphaned statute sections: s34a, s23, s10(m)
    r"^c\d+[a-z]?$",                      # orphaned statute chapters: c90, c94c
    r"^\d+$",                             # purely numeric
    r"^[+-]?\$?\d+$",                     # dollar amounts: $1200, +$1200
    r"^\([a-z0-9]+\)$",                   # parenthetical fragments: (1), (b), (g)
    r"^class [a-e]$",                     # "class a", "class b", etc.
    r"^\d+\s*grams?$",                    # "10 grams", "14 grams"
    r"^\d{1,2}/\d{1,2}$",                # short dates: 6/14, 9/17
    r"^lawpd-qed\.",                      # leftover URLs
    r"^address:\s*",                      # address entries
]

# Exact-match fragments (common broken words)
exact_fragments = {
    "with", "mv with", "law", "grams", "license", "weapon", "vehicle",
    "motor vehicle", "damage", "property", "possess", "mdse", "on", "to",
    "subsq.off", "subsq.off.", "law enf", "yrs whil", "criminal case",
    "statement re:", "viol probate ct", "court warrant", "warrant",
    "lawrence police department", "ti15", "1113", "& s30(1)",
    "-$1200", "+$1200", "$1200", "+60", "500 ft of bldg", "children",
}

total_cleaned = 0
for col in charge_cols:
    col_mask = df[col].notna()
    if not col_mask.any():
        continue
    
    vals = df.loc[col_mask, col].astype(str).str.strip()
    
    # Pattern-based fragment detection
    is_fragment = pd.Series(False, index=vals.index)
    for pattern in fragment_patterns:
        is_fragment = is_fragment | vals.str.match(pattern, case=False, na=False)
    
    # Exact match fragment detection
    is_fragment = is_fragment | vals.str.lower().isin(exact_fragments)
    
    n_cleaned = is_fragment.sum()
    if n_cleaned > 0:
        total_cleaned += n_cleaned
        df.loc[vals.index[is_fragment], col] = np.nan

print(f"Total fragment entries cleaned across all charge columns: {total_cleaned}")

# Show what's left — check for any remaining suspicious short entries
all_remaining = pd.concat([df[c] for c in charge_cols], ignore_index=True).dropna()
short_remaining = all_remaining[all_remaining.str.len() < 10]
print(f"\nRemaining entries under 10 chars (review for any missed fragments):")
print(short_remaining.value_counts().head(30))

Total fragment entries cleaned across all charge columns: 651

Remaining entries under 10 chars (review for any missed fragments):
trespass     146
a&b           71
speeding      56
assault       21
damage to     17
oui drugs      9
viol           9
3rd off.       8
pretense       7
personnel      6
c265 sisa      6
register       5
wanton c       5
keep           5
scheme         5
motor veh      5
violate        5
provide        5
$30(1)         4
stalking       4
1/5/8          3
refuse         3
operation      2
for mv         2
firearm        2
allow          2
drink          2
crimes o       2
improper       2
mayhem         2
Name: count, dtype: int64


### Step 7: Rebuild `cleaned_charges` from clean charge columns
Now that individual `charge_N` columns have been cleaned, reconstruct the semicolon-delimited `cleaned_charges` column from them.

In [23]:
# Rebuild cleaned_charges from the now-clean charge_N columns
charge_cols = [c for c in df.columns if re.match(r"^charge_\d+$", c)]

def rebuild_charges(row):
    parts = [row[c] for c in charge_cols if pd.notna(row[c]) and str(row[c]).strip()]
    return "; ".join(parts) if parts else np.nan

df["cleaned_charges"] = df[charge_cols].apply(rebuild_charges, axis=1)

print(f"Non-null cleaned_charges: {df['cleaned_charges'].notna().sum()}")
print(f"\nTop 20 most common cleaned charges:")
print(df["cleaned_charges"].value_counts().head(20))
print(f"\nTotal unique: {df['cleaned_charges'].nunique()}")

Non-null cleaned_charges: 4546

Top 20 most common cleaned charges:
cleaned_charges
drug, possess class a                               122
sexual conduct for fee                               94
unlicensed operation of mv                           69
disorderly conduct                                   67
drug, possess class b                                67
license suspended, op mv with                        66
any person trafficks fentanyl more than 10 grams     51
oui liquor                                           50
drug, distribute class a                             48
fugitive from justice on court warrant               48
trespass                                             46
drinking in public                                   45
trafficking class a: 28+ grams                       38
drug, possess class a; drug, possess class b         35
motor veh, receive stolen                            33
shoplifting by asportation                           31
assault w/dangerous 

### Step 8: Save unique charges reference v2 and overwrite checkpoint 13

In [24]:
# Build unique charges reference v2
charge_cols = [c for c in df.columns if re.match(r"^charge_\d+$", c)]
all_charges_v2 = pd.concat([df[c] for c in charge_cols], ignore_index=True).dropna()

charge_counts_v2 = all_charges_v2.value_counts().reset_index()
charge_counts_v2.columns = ["charge", "count"]

UNIQUE_CHARGES_V2_PATH = os.path.join(CHARGES_DIR, "unique_charges_reference_v2.csv")
charge_counts_v2.to_csv(UNIQUE_CHARGES_V2_PATH, index=False)

print(f"Total unique charges (v2): {len(charge_counts_v2)}")
print(f"\nTop 30 charges by frequency:")
print(charge_counts_v2.head(30).to_string(index=False))
print(f"\nSaved to {UNIQUE_CHARGES_V2_PATH}")

Total unique charges (v2): 1283

Top 30 charges by frequency:
                                          charge  count
                      unlicensed operation of mv    402
                              disorderly conduct    362
                   license suspended, op mv with    337
                                resisting arrest    291
                           drug, possess class a    274
                           drug, possess class b    223
                                      oui liquor    162
                             stop/yield, fail to    150
                                        trespass    146
any person trafficks fentanyl more than 10 grams    121
                      assault w/dangerous weapon    121
          fugitive from justice on court warrant    120
                        drug, distribute class a    108
                      unregistered motor vehicle    105
             reckless operation of motor vehicle    103
  default warrant: license suspended, op m

In [25]:
# Overwrite checkpoint 13 with cleaned data
df_out = df_full.copy()

new_cols = ["statutes", "cleaned_charges"] + [c for c in df.columns if re.match(r"^(charge|statute)_\d+$", c)]
for col in new_cols:
    df_out[col] = np.nan

mask = df_full["Charges"].notna() & (df_full["Charges"].astype(str).str.strip() != "")
valid_indices = df_full[mask].index

for col in new_cols:
    if col in df.columns:
        df_out.loc[valid_indices, col] = df[col].values

df_out.to_csv(OUT_PATH, index=False)

print("Output shape:", df_out.shape)
print(f"Non-null cleaned_charges: {df_out['cleaned_charges'].notna().sum()}")
print(f"\nOverwritten: {OUT_PATH}")

C:\Users\Indel\AppData\Local\Temp\ipykernel_34264\353618970.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... 'c266 s28' 'c266 s28'
 'c266 s28; c268 s34a; c276 s19; c90 s23; c266 s28; c94c s34; c266 s30a; c266 s28; c90 s10; c90 s23']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[valid_indices, col] = df[col].values
C:\Users\Indel\AppData\Local\Temp\ipykernel_34264\353618970.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['a&b on family / household member / intimate partne; strangulation or suffocation'
 'a&b on family / household member / intimate partne'
 'a&b domestic no 209a in effect' ... 'motor veh, malicious damage to'
 'motor veh, receive stolen'
 "motor veh, receive stolen; false id info, arrst furni to law enf; fugitive from justice

Output shape: (424273, 69)
Non-null cleaned_charges: 4546

Overwritten: c:\Users\Indel\Documents\gatewayinitiative-lawrencepd\scripts\..\data\checkpoints\checkpoint13_cleaned_charges.csv


### Iteration 2 Summary

In [26]:
# Compare v1 vs v2
v1 = pd.read_csv(UNIQUE_CHARGES_PATH)
v2 = charge_counts_v2

print("=== Iteration 2 Summary ===")
print(f"Unique charges v1: {len(v1)}")
print(f"Unique charges v2: {len(v2)}")
print(f"Reduction: {len(v1) - len(v2)} entries removed")
print(f"\nFiles saved:")
print(f"  - {OUT_PATH} (overwritten)")
print(f"  - {UNIQUE_CHARGES_V2_PATH} (new)")

# Show entries in v1 that are NOT in v2 (what we cleaned out)
removed = set(v1["charge"].tolist()) - set(v2["charge"].tolist())
print(f"\nSample entries removed (first 30):")
for entry in sorted(removed)[:30]:
    print(f"  - {entry}")

=== Iteration 2 Summary ===
Unique charges v1: 1760
Unique charges v2: 1283
Reduction: 477 entries removed

Files saved:
  - c:\Users\Indel\Documents\gatewayinitiative-lawrencepd\scripts\..\data\checkpoints\checkpoint13_cleaned_charges.csv (overwritten)
  - c:\Users\Indel\Documents\gatewayinitiative-lawrencepd\scripts\..\data\charges\unique_charges_reference_v2.csv (new)

Sample entries removed (first 30):
  - $1200
  - $1200 (g)
  - & s30(1)
  - (1)
  - (b)
  - (g)
  - (i)
  - +$1200
  - +60
  - -$1200
  - 1/13
  - 1/14
  - 1/16
  - 1/21
  - 10 grams
  - 10/18
  - 10/19
  - 10/26
  - 11/14
  - 11/16
  - 11/17
  - 11/19
  - 11/21
  - 1113
  - 12/16
  - 12/18
  - 13/13
  - 13/14
  - 13/16
  - 13/19


---
## Iteration 3: Clean remaining fragments and junk

After reviewing `unique_charges_reference_v2.csv`, we still have ~100+ problematic entries:

1. **Truncated charge fragments** — partial charge names like `intimate partne`, `damage to`, `malicious c`, `than 10 grams`, `personnel`, `viol`, `wanton c`, `pretense`, etc.
2. **Statute-only fragments** — `c265 sisa`, `$30(1)`, `c269 sioe`, `s14.03`, `c94c s$32e`
3. **Date/time junk** — `7/7124, 1:06 am`, `1/5/8` (corrupted dates)
4. **Generic single words** — `register`, `keep`, `scheme`, `violate`, `provide`, `refuse`, `allow`, `firearm`, `drink`, `use`
5. **System placeholders** — `transferred in [dummy offens`

**Strategy:** Build a comprehensive list of known junk patterns and exact-match fragments, clean them from all `charge_N` columns, then rebuild `cleaned_charges` and save v3.

### Step 1: Define comprehensive fragment patterns and exact matches
Build lists of regex patterns and exact strings that identify junk entries.

In [27]:
# Comprehensive regex patterns for fragments
fragment_patterns_v3 = [
    # Short entries and single chars
    r"^.{1,2}$",                              # 1-2 character entries
    r"^[a-z]$",                               # single letters
    
    # Statute fragments (orphaned sections/chapters)
    r"^[sc]?\d+[a-z]?\s*s[i$]?\d*[a-z]*$",   # statute patterns: c265 sisa, s14.03, etc.
    r"^[sc]\d+[a-z]?$",                       # orphaned chapters: c90, c94c, s14
    r"^\$?\d+\(\d*[a-z]*\)$",                 # $30(1), (1), (b)
    r"^s\d+[a-z]*(\([a-z0-9]+\))?$",          # orphaned sections: s34a, s23
    
    # Numeric and dollar amounts
    r"^[+-]?\$?\d+$",                         # numbers, dollar amounts
    r"^[+-]?\d+\s*grams?$",                   # gram amounts
    r"^\d+\+?\s*grams?$",                     # "10 grams", "28+ grams"
    
    # Dates and times (corrupted or standalone)
    r"^\d{1,2}/\d{1,4}(,?\s*\d{1,2}:\d{2}\s*(am|pm)?)?$",  # dates with optional times
    r"^\d+/\d+/\d+$",                         # standalone dates
    
    # URL/system junk
    r"^lawpd-qed\.",                          # leftover URLs
    r"^address:\s*",                          # address entries
    r"^transferred in \[",                    # system placeholders
    
    # Offense level fragments
    r"^\d+(st|nd|rd|th)\s+off\.?$",           # "2nd off.", "3rd off."
    r"^subsq\.?\s*off\.?$",                   # subsequent offense
    r"^subs\s+off\s*\([a-z]\)$",              # "subs off (f)"
]

# Exact-match fragments (lowercase) — known junk words/phrases
exact_fragments_v3 = {
    # Truncated charge name endings
    "intimate partne", "partne", "intimate pa", "member / intimate partne",
    "household member / intimate pa", "a&b on family / household member /",
    "a&b on family / household", "assault on family / household member / intimate pa",
    "assault on family /", "household member /", "member /",
    
    # Truncated property/damage charges
    "damage to", "malicious c", "wanton c", "wanton property",
    "destruction of prop -$1200,", "destruction of prop +$1200,",
    
    # Truncated drug charges  
    "than 10 grams", "over 14 grams", "class a", "class b", "class c", "class d", "class e",
    "trafficking in", "a, subsq.", "drug, possess to distrib class",
    
    # Truncated weapon charges
    "large cap", "weapon +65", "firearm", "crimes o",
    
    # Truncated warrant fragments
    "conceal id", "personnel", "official, intimidate", "intimdt agg",
    
    # Generic single words (not valid standalone charges)
    "viol", "keep", "register", "scheme", "violate", "provide", "refuse",
    "allow", "drink", "use", "pretense", "operation", "dwelling",
    "improper", "negligent", "stealing", "stealing parts", "asportation",
    "custodian", "cart", "off.", "2nd off.", "3rd off.", "regulation",
    "suffocation", "school/park", "person in fear", "single scheme",
    "under $1200", "for mv", "operate mv with", "oper mv with", "motor veh",
    "with, subsq.off", "+$1200, subsq", "compensation to obtain",
    "grounds,carry", "in mv, drink", "mv, 2nd off", "ist of", "1/5/8",
    "operating, mv, 1st of", "while operating, mv, ist of",
    "obstructed/nontransparent", "displayed", "false/stolen", "subsq. off.",
    "concealing mdse", "race/religion", "order, violate", "mdse, 3rd off.",
    "mdse, 2nd off.", "hypodermic syringe", "distribute", "obtain", "kept",
    
    # Statute-only entries  
    "c265 sisa", "c265 si5b", "c269 sioe", "c94c s$32e", "s14.03", "$30(1)",
    "& s30(1)", "c94c s32e",
    
    # Date/time junk
    "7/7124, 1:06 am", "7/4124, 4:22 am",
    
    # Location/system entries
    "lawrence, ma", "lowell, ma", "methuen, ma",
    "transferred in [dummy offens",
}

print(f"Defined {len(fragment_patterns_v3)} regex patterns")
print(f"Defined {len(exact_fragments_v3)} exact-match fragments")

Defined 17 regex patterns
Defined 105 exact-match fragments


### Step 2: Apply fragment cleanup to all charge columns
Scan all `charge_N` columns and set matching fragments to NaN.

In [28]:
charge_cols = [c for c in df.columns if re.match(r"^charge_\d+$", c)]

total_cleaned_v3 = 0
cleaned_examples = []

for col in charge_cols:
    col_mask = df[col].notna()
    if not col_mask.any():
        continue
    
    vals = df.loc[col_mask, col].astype(str).str.strip()
    
    # Pattern-based fragment detection
    is_fragment = pd.Series(False, index=vals.index)
    for pattern in fragment_patterns_v3:
        is_fragment = is_fragment | vals.str.match(pattern, case=False, na=False)
    
    # Exact match fragment detection (case-insensitive)
    is_fragment = is_fragment | vals.str.lower().isin(exact_fragments_v3)
    
    n_cleaned = is_fragment.sum()
    if n_cleaned > 0:
        # Collect examples of what we're cleaning
        examples = vals[is_fragment].value_counts().head(5).index.tolist()
        cleaned_examples.extend(examples)
        
        total_cleaned_v3 += n_cleaned
        df.loc[vals.index[is_fragment], col] = np.nan

print(f"Total fragment entries cleaned in iteration 3: {total_cleaned_v3}")
print(f"\nSample entries removed (first 30 unique):")
for entry in sorted(set(cleaned_examples))[:30]:
    print(f"  - {entry}")

Total fragment entries cleaned in iteration 3: 284

Sample entries removed (first 30 unique):
  - $30(1)
  - 1/5/8
  - 3rd off.
  - assault on family / household member / intimate pa
  - damage to
  - drink
  - for mv
  - intimate partne
  - ist of
  - lawrence, ma
  - malicious c
  - mdse, 3rd off.
  - member / intimate partne
  - obtain
  - order, violate
  - partne
  - pretense
  - school/park
  - subs off (f)
  - subsq. off.
  - than 10 grams
  - viol
  - violate
  - wanton property


### Step 3: Review remaining short entries
Check if any suspicious entries remain after cleanup.

In [29]:
# Review remaining short entries
all_remaining_v3 = pd.concat([df[c] for c in charge_cols], ignore_index=True).dropna()
short_remaining_v3 = all_remaining_v3[all_remaining_v3.str.len() < 15]

print(f"Remaining entries under 15 chars:")
print(short_remaining_v3.value_counts().head(40))

print(f"\n--- Entries that look like valid charges (keeping these): ---")
valid_short = ["trespass", "a&b", "speeding", "assault", "oui drugs", "stalking", 
               "mayhem", "conspiracy", "murder", "kidnapping", "carjacking"]
for v in valid_short:
    count = (all_remaining_v3.str.lower() == v.lower()).sum()
    if count > 0:
        print(f"  {v}: {count}")

Remaining entries under 15 chars:
oui liquor        162
trespass          146
a&b                71
speeding           56
assault            21
identity fraud     18
robbery, armed     18
oui drugs           9
conspiracy          7
turn, improper      7
home invasion       4
stalking            4
intimidate          4
carjacking          2
kidnapping          2
mayhem              2
murder              1
violation           1
manslaughter        1
Name: count, dtype: int64

--- Entries that look like valid charges (keeping these): ---
  trespass: 146
  a&b: 71
  speeding: 56
  assault: 21
  oui drugs: 9
  stalking: 4
  mayhem: 2
  conspiracy: 7
  murder: 1
  kidnapping: 2
  carjacking: 2


### Step 4: Rebuild `cleaned_charges` from clean charge columns
Reconstruct the semicolon-delimited `cleaned_charges` column from the now-clean `charge_N` columns.

In [30]:
# Rebuild cleaned_charges from the now-clean charge_N columns
def rebuild_charges_v3(row):
    parts = [row[c] for c in charge_cols if pd.notna(row[c]) and str(row[c]).strip()]
    return "; ".join(parts) if parts else np.nan

df["cleaned_charges"] = df[charge_cols].apply(rebuild_charges_v3, axis=1)

print(f"Non-null cleaned_charges: {df['cleaned_charges'].notna().sum()}")
print(f"\nTop 20 most common cleaned charges:")
print(df["cleaned_charges"].value_counts().head(20))
print(f"\nTotal unique cleaned_charges: {df['cleaned_charges'].nunique()}")

Non-null cleaned_charges: 4542

Top 20 most common cleaned charges:
cleaned_charges
drug, possess class a                               122
sexual conduct for fee                               94
unlicensed operation of mv                           69
drug, possess class b                                67
disorderly conduct                                   67
license suspended, op mv with                        66
any person trafficks fentanyl more than 10 grams     51
oui liquor                                           50
fugitive from justice on court warrant               48
drug, distribute class a                             48
drinking in public                                   46
trespass                                             46
trafficking class a: 28+ grams                       38
drug, possess class a; drug, possess class b         35
motor veh, receive stolen                            33
shoplifting by asportation                           31
assault w/dangerous 

### Step 5: Save unique charges reference v3 and update checkpoint 13

In [31]:
# Build unique charges reference v3
all_charges_v3 = pd.concat([df[c] for c in charge_cols], ignore_index=True).dropna()

charge_counts_v3 = all_charges_v3.value_counts().reset_index()
charge_counts_v3.columns = ["charge", "count"]

UNIQUE_CHARGES_V3_PATH = os.path.join(CHARGES_DIR, "unique_charges_reference_v3.csv")
charge_counts_v3.to_csv(UNIQUE_CHARGES_V3_PATH, index=False)

print(f"Total unique charges (v3): {len(charge_counts_v3)}")
print(f"\nTop 30 charges by frequency:")
print(charge_counts_v3.head(30).to_string(index=False))
print(f"\nSaved to {UNIQUE_CHARGES_V3_PATH}")

Total unique charges (v3): 1194

Top 30 charges by frequency:
                                          charge  count
                      unlicensed operation of mv    402
                              disorderly conduct    362
                   license suspended, op mv with    337
                                resisting arrest    291
                           drug, possess class a    274
                           drug, possess class b    223
                                      oui liquor    162
                             stop/yield, fail to    150
                                        trespass    146
any person trafficks fentanyl more than 10 grams    121
                      assault w/dangerous weapon    121
          fugitive from justice on court warrant    120
                        drug, distribute class a    108
                      unregistered motor vehicle    105
             reckless operation of motor vehicle    103
  default warrant: license suspended, op m

In [32]:
# Update checkpoint 13 with v3 cleaned data
df_out = df_full.copy()

new_cols = ["statutes", "cleaned_charges"] + [c for c in df.columns if re.match(r"^(charge|statute)_\d+$", c)]
for col in new_cols:
    df_out[col] = np.nan

mask = df_full["Charges"].notna() & (df_full["Charges"].astype(str).str.strip() != "")
valid_indices = df_full[mask].index

for col in new_cols:
    if col in df.columns:
        df_out.loc[valid_indices, col] = df[col].values

df_out.to_csv(OUT_PATH, index=False)

print("Output shape:", df_out.shape)
print(f"Non-null cleaned_charges: {df_out['cleaned_charges'].notna().sum()}")
print(f"\nOverwritten: {OUT_PATH}")

C:\Users\Indel\AppData\Local\Temp\ipykernel_34264\4249960986.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... 'c266 s28' 'c266 s28'
 'c266 s28; c268 s34a; c276 s19; c90 s23; c266 s28; c94c s34; c266 s30a; c266 s28; c90 s10; c90 s23']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[valid_indices, col] = df[col].values
C:\Users\Indel\AppData\Local\Temp\ipykernel_34264\4249960986.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['a&b on family / household member / intimate partne; strangulation or suffocation'
 'a&b on family / household member / intimate partne'
 'a&b domestic no 209a in effect' ... 'motor veh, malicious damage to'
 'motor veh, receive stolen'
 "motor veh, receive stolen; false id info, arrst furni to law enf; fugitive from justi

Output shape: (424273, 69)
Non-null cleaned_charges: 4542

Overwritten: c:\Users\Indel\Documents\gatewayinitiative-lawrencepd\scripts\..\data\checkpoints\checkpoint13_cleaned_charges.csv


### Iteration 3 Summary

In [33]:
# Compare v2 vs v3
v2 = pd.read_csv(os.path.join(CHARGES_DIR, "unique_charges_reference_v2.csv"))
v3 = charge_counts_v3

print("=== Iteration 3 Summary ===")
print(f"Unique charges v2: {len(v2)}")
print(f"Unique charges v3: {len(v3)}")
print(f"Reduction: {len(v2) - len(v3)} entries removed")
print(f"Fragment entries cleaned: {total_cleaned_v3}")
print(f"\nFiles saved:")
print(f"  - {OUT_PATH} (overwritten)")
print(f"  - {UNIQUE_CHARGES_V3_PATH} (new)")

# Show entries removed in v3
removed_v3 = set(v2["charge"].tolist()) - set(v3["charge"].tolist())
print(f"\nEntries removed in v3 ({len(removed_v3)} total, showing first 40):")
for entry in sorted(removed_v3)[:40]:
    print(f"  - {entry}")

=== Iteration 3 Summary ===
Unique charges v2: 1283
Unique charges v3: 1194
Reduction: 89 entries removed
Fragment entries cleaned: 284

Files saved:
  - c:\Users\Indel\Documents\gatewayinitiative-lawrencepd\scripts\..\data\checkpoints\checkpoint13_cleaned_charges.csv (overwritten)
  - c:\Users\Indel\Documents\gatewayinitiative-lawrencepd\scripts\..\data\charges\unique_charges_reference_v3.csv (new)

Entries removed in v3 (89 total, showing first 40):
  - $30(1)
  - +$1200, subsq
  - 1/5/8
  - 2nd off.
  - 3rd off.
  - 7/4124, 4:22 am
  - 7/7124, 1:06 am
  - a, subsq.
  - allow
  - asportation
  - assault on family / household member / intimate pa
  - c265 si5b
  - c265 sisa
  - c269 sioe
  - c94c s$32e
  - cart
  - compensation to obtain
  - conceal id
  - concealing mdse
  - crimes o
  - custodian
  - damage to
  - displayed
  - distribute
  - drink
  - dwelling
  - false/stolen
  - firearm
  - for mv
  - grounds,carry
  - household member / intimate pa
  - hypodermic syringe
  - imp